In [3]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.8/381.8 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.6 MB/s eta 0:00:00


In [4]:
import numpy as np
from collections import Counter

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke


# ------------------------------------------------
# 1. DNA input
# ------------------------------------------------

dna = "ATGCGTACGTTAGCGTACGATCGTAGCTAGCTTGACGATCGTACGTTAGC"

pairs = [dna[i:i+2] for i in range(0,len(dna),2)]

print("\nDNA pairs:")
print(pairs)


# ------------------------------------------------
# 2. Motif detection
# ------------------------------------------------

motif_candidates = Counter(zip(pairs,pairs[1:]))

top = motif_candidates.most_common(1)

motif=None
if top and top[0][1] > 1:
    motif=top[0][0]

print("\nDetected motif:",motif)


# ------------------------------------------------
# 3. Motif compression
# ------------------------------------------------

compressed=[]
i=0

while i < len(pairs):

    if motif and i < len(pairs)-1 and (pairs[i],pairs[i+1])==motif:
        compressed.append("M1")
        i+=2
    else:
        compressed.append(pairs[i])
        i+=1

print("\nCompressed sequence:")
print(compressed)


# ------------------------------------------------
# 4. Pair encoding
# ------------------------------------------------

pair_map={
'AA':0,'AC':1,'AG':2,'AT':3,
'CA':4,'CC':5,'CG':6,'CT':7,
'GA':8,'GC':9,'GG':10,'GT':11,
'TA':12,'TC':13,'TG':14,'TT':15
}

symbols=[]

for item in compressed:
    if item=="M1":
        symbols.append(16)
    else:
        symbols.append(pair_map[item])

print("\nSymbol sequence:",symbols)


# ------------------------------------------------
# 5. Circuit size
# ------------------------------------------------

n_symbols=len(symbols)

index_qubits=int(np.ceil(np.log2(n_symbols)))
value_qubits=5

total_qubits=index_qubits+value_qubits

print("\nIndex qubits:",index_qubits)
print("Value qubits:",value_qubits)
print("Total qubits:",total_qubits)

qc=QuantumCircuit(total_qubits)


# ------------------------------------------------
# 6. Create superposition of indices
# ------------------------------------------------

for q in range(index_qubits):
    qc.h(q)


# ------------------------------------------------
# 7. Binary-tree style encoding
# ------------------------------------------------

for idx,val in enumerate(symbols):

    idx_bits=format(idx,f"0{index_qubits}b")
    val_bits=format(val,f"0{value_qubits}b")

    for q,b in enumerate(idx_bits):
        if b=="0":
            qc.x(q)

    for j,b in enumerate(val_bits[::-1]):
        if b=="1":
            qc.ccx(0,1,index_qubits+j)

    for q,b in enumerate(idx_bits):
        if b=="0":
            qc.x(q)


print("\nCircuit depth before optimization:",qc.depth())


# ------------------------------------------------
# 8. SHOW DNA REPRESENTATION AS QUBIT STATE
# ------------------------------------------------

state = Statevector.from_instruction(qc)

print("\nQuantum representation of DNA (Dirac notation):\n")

for i,amp in enumerate(state.data):

    if abs(amp)>1e-6:

        bitstring=format(i,f"0{total_qubits}b")

        idx_bits=bitstring[:index_qubits]
        val_bits=bitstring[index_qubits:]

        print(f"{amp:.3f} |{idx_bits}>|{val_bits}>")


# ------------------------------------------------
# 9. Transpiler optimization
# ------------------------------------------------

sim_backend=AerSimulator()

optimized=transpile(
    qc,
    sim_backend,
    optimization_level=3
)

print("\nCircuit depth after optimization:",optimized.depth())


# ------------------------------------------------
# 10. Ideal state
# ------------------------------------------------

ideal_state=Statevector.from_instruction(optimized)


# ------------------------------------------------
# 11. FakeSherbrooke noise model
# ------------------------------------------------

fake_backend=FakeSherbrooke()

noise_model=NoiseModel.from_backend(fake_backend)


# ------------------------------------------------
# 12. Noisy simulation
# ------------------------------------------------

noisy_circuit=optimized.copy()

noisy_circuit.save_density_matrix()

sim=AerSimulator(
    method="density_matrix",
    noise_model=noise_model
)

noisy_circuit=transpile(noisy_circuit,sim)

job=sim.run(noisy_circuit)

result=job.result()

noisy_density=result.data(0)["density_matrix"]


# ------------------------------------------------
# 13. Exact fidelity
# ------------------------------------------------

fidelity=state_fidelity(ideal_state,noisy_density)

print("\nExact state fidelity:",fidelity)


# ------------------------------------------------
# 14. Circuit metrics
# ------------------------------------------------

print("\n--- Circuit Metrics ---")

print("Qubits:",optimized.num_qubits)
print("Depth:",optimized.depth())
print("CNOT count:",optimized.count_ops().get("cx",0))
print("SWAP count:",optimized.count_ops().get("swap",0))


# ------------------------------------------------
# 15. Circuit diagram
# ------------------------------------------------

print("\nCircuit diagram:\n")

print(optimized.draw())


DNA pairs:
['AT', 'GC', 'GT', 'AC', 'GT', 'TA', 'GC', 'GT', 'AC', 'GA', 'TC', 'GT', 'AG', 'CT', 'AG', 'CT', 'TG', 'AC', 'GA', 'TC', 'GT', 'AC', 'GT', 'TA', 'GC']

Detected motif: ('GT', 'AC')

Compressed sequence:
['AT', 'GC', 'M1', 'GT', 'TA', 'GC', 'M1', 'GA', 'TC', 'GT', 'AG', 'CT', 'AG', 'CT', 'TG', 'AC', 'GA', 'TC', 'M1', 'GT', 'TA', 'GC']

Symbol sequence: [3, 9, 16, 11, 12, 9, 16, 8, 13, 11, 2, 7, 2, 7, 14, 1, 8, 13, 16, 11, 12, 9]

Index qubits: 5
Value qubits: 5
Total qubits: 10

Circuit depth before optimization: 88

Quantum representation of DNA (Dirac notation):

0.177+0.000j |00000>|00011>
0.177+0.000j |00000>|00111>
0.177+0.000j |00000>|01011>
0.177+0.000j |00000>|01111>
0.177+0.000j |00000>|10011>
0.177+0.000j |00000>|10111>
0.177+0.000j |00000>|11011>
0.177+0.000j |00000>|11111>
0.177+0.000j |01001>|00010>
0.177+0.000j |01001>|00110>
0.177+0.000j |01001>|01010>
0.177+0.000j |01001>|01110>
0.177+0.000j |01001>|10010>
0.177+0.000j |01001>|10110>
0.177+0.000j |01001>|1101